# O04_ef35_c05 histinit パス可視化

この notebook は `histinit` 配下の生成結果を見るためのものです。

主な使い方:

1. `SEED = "seed03"` のように60年パスを指定する
2. `CHUNK = 4` のように12分割された5年程度の区間を指定する
3. 60年全体の overview と、指定区間の詳細グラフを見る

ディレクトリ構成:

- `60y_paths/generated_paths_seed01.csv` 〜 `generated_paths_seed10.csv`
- `5y_paths/generated_paths_seed01-1.csv` 〜 `generated_paths_seed10-12.csv`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd()
if ROOT.name != "histinit":
    CAND = Path("/home/u00121/graph_ssm_abm/ZA_goal09_variant_ZA_minimal_final/results_fix4_7/O04_ef35_c05/histinit")
    if CAND.exists():
        ROOT = CAND

PATH60 = ROOT / "60y_paths"
PATH5 = ROOT / "5y_paths"
REAL_PATH = Path("/home/u00121/output.csv")

print("ROOT:", ROOT)
print("60y files:", len(list(PATH60.glob("generated_paths_seed*.csv"))))
print("5y files:", len(list(PATH5.glob("generated_paths_seed*.csv"))))


## 0. パス指定

ここだけ変えれば、見る対象を切り替えられます。

- `SEED`: `seed01`〜`seed10`
- `CHUNK`: 1〜12


In [ ]:
SEED = "seed01"
CHUNK = 1

ROLLING_CORR_WINDOW = 90
ROLLING_VOL_WINDOW = 21

p60 = PATH60 / f"generated_paths_{SEED}.csv"
p5 = PATH5 / f"generated_paths_{SEED}-{CHUNK}.csv"

assert p60.exists(), p60
assert p5.exists(), p5

df60 = pd.read_csv(p60, parse_dates=["Date"])
df5 = pd.read_csv(p5, parse_dates=["Date"])
real = pd.read_csv(REAL_PATH, parse_dates=["Date"]).tail(len(df60)).reset_index(drop=True)

print("60y:", p60.name, len(df60), df60["Date"].iloc[0].date(), "→", df60["Date"].iloc[-1].date())
print("5y :", p5.name, len(df5), df5["Date"].iloc[0].date(), "→", df5["Date"].iloc[-1].date())


## 1. 10本の60年パス一覧

10本全体の水準推移をざっと見るセルです。黒線は実データです。

In [ ]:
files60 = sorted(PATH60.glob("generated_paths_seed*.csv"))
summary = []
for p in files60:
    d = pd.read_csv(p, parse_dates=["Date"])
    summary.append({
        "seed": p.stem.replace("generated_paths_", ""),
        "rows": len(d),
        "start": d["Date"].iloc[0].date(),
        "end": d["Date"].iloc[-1].date(),
        "sp500_start": d["sp500_abs"].iloc[0],
        "sp500_end": d["sp500_abs"].iloc[-1],
        "dgs10_start": d["DGS10_abs"].iloc[0],
        "dgs10_end": d["DGS10_abs"].iloc[-1],
        "sp500_std": d["sp500"].std(),
        "sp500_kurt": d["sp500"].kurt(),
    })
summary_df = pd.DataFrame(summary)
display(summary_df)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(real["Date"], real["sp500_abs"], color="black", lw=1.6, label="real")
axes[1].plot(real["Date"], real["DGS10_abs"], color="black", lw=1.6, label="real")
for p in files60:
    d = pd.read_csv(p, parse_dates=["Date"])
    label = p.stem.replace("generated_paths_", "")
    axes[0].plot(d["Date"], d["sp500_abs"], lw=0.9, alpha=0.8, label=label)
    axes[1].plot(d["Date"], d["DGS10_abs"], lw=0.9, alpha=0.8, label=label)
axes[0].set_title("60y paths: SP500 level")
axes[1].set_title("60y paths: DGS10 level")
axes[0].set_ylabel("SP500")
axes[1].set_ylabel("DGS10 (%)")
for ax in axes:
    ax.grid(alpha=0.3)
axes[1].legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()


## 2. 指定SEEDの60年 overview

以前の `paths_overview` に近い見方です。価格水準、日次変化、絶対リターン、ローリング相関をまとめて表示します。

In [ ]:
rc90 = df60["sp500"].rolling(ROLLING_CORR_WINDOW).corr(df60["DGS10"])
vol21 = df60["sp500"].rolling(ROLLING_VOL_WINDOW).std()

fig, axes = plt.subplots(4, 2, figsize=(15, 12), sharex=True)
axes = axes.ravel()

axes[0].plot(real["Date"], real["sp500_abs"], color="gray", lw=1.0, label="real")
axes[0].plot(df60["Date"], df60["sp500_abs"], color="tab:blue", lw=1.1, label=SEED)
axes[0].set_title("SP500 level")
axes[0].legend()

axes[1].plot(real["Date"], real["DGS10_abs"], color="gray", lw=1.0, label="real")
axes[1].plot(df60["Date"], df60["DGS10_abs"], color="tab:red", lw=1.1, label=SEED)
axes[1].set_title("DGS10 level")
axes[1].legend()

axes[2].plot(df60["Date"], df60["sp500"], color="tab:blue", lw=0.55)
axes[2].set_title("SP500 daily return")

axes[3].plot(df60["Date"], df60["DGS10"], color="tab:red", lw=0.55)
axes[3].set_title("DGS10 daily change")

axes[4].plot(df60["Date"], df60["sp500"].abs(), color="tab:orange", lw=0.6)
axes[4].set_title("|SP500 return|")

axes[5].plot(df60["Date"], vol21, color="purple", lw=0.8)
axes[5].set_title(f"SP500 rolling volatility ({ROLLING_VOL_WINDOW}d)")

axes[6].plot(df60["Date"], rc90, color="black", lw=0.8)
axes[6].axhline(0, color="gray", lw=0.8)
axes[6].set_ylim(-1, 1)
axes[6].set_title(f"{ROLLING_CORR_WINDOW}d rolling corr(SP500 return, DGS10 change)")

axes[7].hist(df60["sp500"], bins=120, color="tab:blue", alpha=0.65, density=True)
axes[7].set_title("SP500 return histogram")

for ax in axes:
    ax.grid(alpha=0.3)
plt.suptitle(f"60y overview: {SEED}", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print(df60[["sp500", "DGS10"]].describe())
print("SP500 kurtosis(excess):", df60["sp500"].kurt())
print("DGS10 kurtosis(excess):", df60["DGS10"].kurt())


## 3. 指定CHUNKの5年区間 overview

`SEED` と `CHUNK` で指定した部分パスだけを詳しく見ます。

In [ ]:
rc90_5 = df5["sp500"].rolling(ROLLING_CORR_WINDOW).corr(df5["DGS10"])
vol21_5 = df5["sp500"].rolling(ROLLING_VOL_WINDOW).std()

fig, axes = plt.subplots(4, 2, figsize=(15, 12), sharex=True)
axes = axes.ravel()

axes[0].plot(df5["Date"], df5["sp500_abs"], color="tab:blue", lw=1.1)
axes[0].set_title("SP500 level")

axes[1].plot(df5["Date"], df5["DGS10_abs"], color="tab:red", lw=1.1)
axes[1].set_title("DGS10 level")

axes[2].plot(df5["Date"], df5["sp500"], color="tab:blue", lw=0.7)
axes[2].set_title("SP500 daily return")

axes[3].plot(df5["Date"], df5["DGS10"], color="tab:red", lw=0.7)
axes[3].set_title("DGS10 daily change")

axes[4].plot(df5["Date"], df5["sp500"].abs(), color="tab:orange", lw=0.8)
axes[4].set_title("|SP500 return|")

axes[5].plot(df5["Date"], vol21_5, color="purple", lw=0.9)
axes[5].set_title(f"SP500 rolling volatility ({ROLLING_VOL_WINDOW}d)")

axes[6].plot(df5["Date"], rc90_5, color="black", lw=0.9)
axes[6].axhline(0, color="gray", lw=0.8)
axes[6].set_ylim(-1, 1)
axes[6].set_title(f"{ROLLING_CORR_WINDOW}d rolling corr")

axes[7].hist(df5["sp500"], bins=60, color="tab:blue", alpha=0.65, density=True)
axes[7].set_title("SP500 return histogram")

for ax in axes:
    ax.grid(alpha=0.3)
plt.suptitle(f"5y-ish chunk overview: {SEED}-{CHUNK}", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print(df5[["Date", "sp500_abs", "DGS10_abs", "sp500", "DGS10"]].head())
print(df5[["Date", "sp500_abs", "DGS10_abs", "sp500", "DGS10"]].tail())
print(df5[["sp500", "DGS10"]].describe())


## 4. 指定SEEDの12分割を一覧

指定した `SEED` について、12個の5年区間の水準推移を小分けで見ます。

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(15, 18), sharey=False)
axes = axes.ravel()
for j in range(1, 13):
    p = PATH5 / f"generated_paths_{SEED}-{j}.csv"
    d = pd.read_csv(p, parse_dates=["Date"])
    ax = axes[j-1]
    ax2 = ax.twinx()
    ax.plot(d["Date"], d["sp500_abs"], color="tab:blue", lw=0.9)
    ax2.plot(d["Date"], d["DGS10_abs"], color="tab:red", lw=0.8, alpha=0.75)
    ax.set_title(f"{SEED}-{j}: {d['Date'].iloc[0].date()} → {d['Date'].iloc[-1].date()}")
    ax.grid(alpha=0.25)
plt.suptitle(f"All chunks for {SEED}: blue=SP500, red=DGS10", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()


## 5. 分割メタ情報

In [ ]:
meta_path = ROOT / "split_metadata.csv"
if meta_path.exists():
    meta = pd.read_csv(meta_path)
    display(meta[meta["seed_label"].eq(SEED)].reset_index(drop=True))
else:
    print("split_metadata.csv がありません。")
